In [ ]:
# [optional] Return to the notebook folder
#cd <>/workflow_initiation/

In [ ]:
# [optional] Cleaning if needed to reset the noebook
cd "$(dirname "$JPY_SESSION_NAME")"
rm -rf Data/ logs FastQC multiqc_data dag.pdf multiqc_report.html

# Snakemake initiation

## Outline

- Explore a smk workflow: [smk_preprocessing](https://gitlab.com/ifb-elixirfr/training/workflows/smk_preprocessing)
- Run a smk workflow
- Manage the dependency


## Notebook setup

Data used for this training are available on Zenodo : https://zenodo.org/record/3997237

You don't need to download them, they are already available on the Wave Research cluster.

You just have to create a link to the data in your snakemake folder.

In [ ]:
cp -r /projects/ADVANCED/snakemake/Data Data

---

## Explore a smk workflow

We will use the workflow [smk_preprocessing](https://gitlab.com/ifb-elixirfr/training/workflows/smk_preprocessing)

In [ ]:
# Pin snakemake versions used below
SNAKEMAKE_VERSION=7.32.4
# Ignore Python warning about future deprecation warning
export PYTHONWARNINGS="ignore::FutureWarning"

In [ ]:
tree

In [ ]:
cat Snakefile

### Create a DAG of your workflow

In [ ]:
module load snakemake/$SNAKEMAKE_VERSION graphviz

snakemake --dag | dot -Tpdf > dag.pdf

This DAG is rather small but for a more complex workflow, it could show the power of the workflow engine usage.

## Run a smk workflow
### "Locally"

In [ ]:
module purge
module load bioinfo-shared
module load snakemake/$SNAKEMAKE_VERSION

snakemake --cores 1 --use-envmodules

#### Use profile

How to adapt the amount of resources, RAM, time, and the number of threads to a particular job through the profile config.yaml file.

In [ ]:
cat profile/default/config.yaml

In [ ]:
module purge
module load bioinfo-shared
module load snakemake/$SNAKEMAKE_VERSION

snakemake --profile ./profile/default/ --use-envmodules

### On a HPC cluster (SLURM)

There are 3 options to run a worklow on a SLURM cluster:
 - Generic cluster support
   - `--cluster "sbatch"`: 
     - smk will launch independant sbatch jobs
     - Less control on the jobs
   - `--drmaa` [DRMAA](https://www.drmaa.org) / Distributed Resource Management Application API
     - A API specification for tightly coupled and portable programmatic access to cluster, grid, and cloud systems: SGE, PBS, Condor, SLURM ...
     - If available, DRMAA is preferable over the generic cluster modes because it provides better control and error handling.
 - [SLURM dedicated backend](https://snakemake.readthedocs.io/en/v7.19.1/executing/cluster.html#executing-on-slurm-clusters) (since 7.19.0)
   - `--slurm`

#### A dedicated profile

In [ ]:
cat profile/slurm/config.yaml

In [ ]:
rm -rf multiqc_data FastQC multiqc_report.html

module purge
module load bioinfo-shared
module load snakemake/$SNAKEMAKE_VERSION singularity

unset SLURM_EXPORT_ENV

snakemake --slurm --profile ./profile/slurm --use-envmodules

In [ ]:
sacct -X
jobs

### Rerun

In [ ]:
module purge
module load bioinfo-shared
module load snakemake/$SNAKEMAKE_VERSION
snakemake --slurm --profile ./profile/slurm --use-envmodules

> Nothing to be done (all requested files are present and up to date).

#### What about adding new input data?

🫵 Add the following samples in our analyse:

```
SRR3105697_chr18
SRR3105698_chr18
SRR3105699_chr18
```

In [ ]:
snakemake --slurm --profile ./profile/slurm --use-envmodules

🫵 Questions:

- What happened?

#### Let change one input file

In [ ]:
touch Data/SRR3099585_chr18.fastq.gz
snakemake --slurm --profile ./profile/slurm --use-envmodules

> reason: Updated input files: Data/SRR3099585_chr18.fastq.gz

As consequence, all the downstream rule that involved `Data/SRR3099585_chr18.fastq.gz` are rerun:
 - 1 of the 4 fasqtc
 - the multiqc

## Manage the dependency

```yaml
rule multiqc:
  output:
    [...]
  input:
    [...]
  conda:
    "envs/multiqc-1.19.yml"
  container:
    "docker://quay.io/biocontainers/multiqc:1.19--pyhdfd78af_0"
  envmodules:
    "bioinfo-shared",
    "multiqc/1.19"
  shell: [...]
```

### Singularity (Apptainer)

In this case, Singularity can use:
 - A docker container URL `docker://...` that will be pulled and converted in a `.sif` singularity image.
 - A Singularity `.sif` image local or only `https://...`

In [ ]:
module purge;

module load snakemake/$SNAKEMAKE_VERSION singularity
snakemake --slurm --profile ./profile/slurm --use-singularity

### Conda

In [ ]:
cat envs/multiqc-1.19.yml

In [ ]:
module purge; 

module load snakemake/$SNAKEMAKE_VERSION conda
snakemake --slurm --profile ./profile/slurm --use-conda